In [8]:
import math
import numpy as np
import scipy.stats as stats
from scipy.special import erfc

# ------------------------------------------------------------
# 1. Frequency (Monobit) Test
def monobit_test(bits):
    n = len(bits)
    S = sum(1 if b == 1 else -1 for b in bits)
    S_obs = abs(S) / math.sqrt(n)
    p = erfc(S_obs / math.sqrt(2))
    return p

# 2. Frequency Test within a Block
def block_frequency_test(bits, M=128):
    n = len(bits)
    N = n // M
    if N == 0:
        return None
    proportions = []
    for i in range(N):
        block = bits[i*M : (i+1)*M]
        pi = sum(block) / M
        proportions.append(pi)
    chi_sq = 4 * M * sum((pi - 0.5)**2 for pi in proportions)
    p = stats.chi2.sf(chi_sq, N)
    return p

# 3. Runs Test
def runs_test(bits):
    n = len(bits)
    pi = sum(bits) / n
    if abs(pi - 0.5) >= (2 / math.sqrt(n)):
        return 0.0
    runs = 1
    for i in range(1, n):
        if bits[i] != bits[i-1]:
            runs += 1
    num = runs - 2 * n * pi * (1 - pi)
    den = 2 * math.sqrt(n) * pi * (1 - pi)
    if den == 0:
        return 0.0
    p = erfc(abs(num / den) / math.sqrt(2))
    return p

# 4. Longest Run of Ones in a Block (para M=10000)
def longest_run_ones_test(bits, M=10000):
    n = len(bits)
    if n < M:
        return None
    N = n // M
    # bins para M=10000 (según NIST)
    bins = [0]*7
    for i in range(N):
        block = bits[i*M : (i+1)*M]
        max_run = 0
        cur = 0
        for b in block:
            if b == 1:
                cur += 1
                max_run = max(max_run, cur)
            else:
                cur = 0
        if max_run <= 10:
            bins[0] += 1
        elif max_run == 11:
            bins[1] += 1
        elif max_run == 12:
            bins[2] += 1
        elif max_run == 13:
            bins[3] += 1
        elif max_run == 14:
            bins[4] += 1
        elif max_run == 15:
            bins[5] += 1
        else:
            bins[6] += 1
    # Proporciones esperadas (NIST)
    pi = [0.0882, 0.2092, 0.2483, 0.1933, 0.1208, 0.0675, 0.0727]
    chi_sq = sum((bins[i] - N*pi[i])**2 / (N*pi[i]) for i in range(7))
    p = stats.chi2.sf(chi_sq, 6)
    return p

# 5. Binary Matrix Rank Test (con eliminación Gaussiana en GF2)
def gf2_rank(matrix):
    """Calcula el rango de una matriz binaria sobre GF(2)."""
    A = matrix.astype(int).copy()
    rows, cols = A.shape
    rank = 0
    for col in range(cols):
        pivot = None
        for row in range(rank, rows):
            if A[row, col] == 1:
                pivot = row
                break
        if pivot is None:
            continue
        A[[rank, pivot]] = A[[pivot, rank]]
        for row in range(rows):
            if row != rank and A[row, col] == 1:
                A[row, :] ^= A[rank, :]
        rank += 1
        if rank == rows:
            break
    return rank

def binary_matrix_rank_test(bits, m=32, q=32):
    n = len(bits)
    N = n // (m*q)
    if N == 0:
        return None
    full_rank = 0
    full_minus_one = 0
    other = 0
    max_rank = min(m, q)
    for i in range(N):
        block = bits[i*m*q : (i+1)*m*q]
        matrix = np.array(block).reshape(m, q)
        rank = gf2_rank(matrix)
        if rank == max_rank:
            full_rank += 1
        elif rank == max_rank - 1:
            full_minus_one += 1
        else:
            other += 1
    pi_full = 0.2888
    pi_minus_one = 0.5776
    pi_other = 0.1336
    chi_sq = ((full_rank - N*pi_full)**2/(N*pi_full) +
              (full_minus_one - N*pi_minus_one)**2/(N*pi_minus_one) +
              (other - N*pi_other)**2/(N*pi_other))
    p = stats.chi2.sf(chi_sq, 2)
    return p

# 6. Discrete Fourier Transform (Spectral) Test
def dft_test(bits):
    n = len(bits)
    X = np.array([1 if b == 1 else -1 for b in bits])
    fft = np.fft.fft(X)
    mags = np.abs(fft[:n//2])
    T = np.sqrt(np.log(1/0.05) * n)   # umbral 95%
    N1 = np.sum(mags < T)
    d = (N1 - 0.95*n/2) / np.sqrt(0.95*0.05*n/2)
    p = erfc(abs(d) / np.sqrt(2))
    return p

# 7. Non-overlapping Template Matching Test (plantilla fija)
def non_overlapping_template_test(bits, m=9, template='000000001'):
    n = len(bits)
    M = 1000
    N = n // M
    if N == 0:
        return None
    template_bits = [int(c) for c in template]
    mu = (M - m + 1) / (2**m)
    sigma2 = M * (1/(2**m) - (2*m - 1)/(2**(2*m)))
    W = []
    for i in range(N):
        block = bits[i*M:(i+1)*M]
        count = 0
        pos = 0
        while pos <= M - m:
            if block[pos:pos+m] == template_bits:
                count += 1
                pos += m   # no solapamiento
            else:
                pos += 1
        W.append(count)
    chi_sq = sum(((w - mu)**2) / sigma2 for w in W)
    p = stats.chi2.sf(chi_sq, N)
    return p

# 8. Overlapping Template Matching Test (simplificado, plantilla fija)
def overlapping_template_test(bits, m=9, template='000000001'):
    n = len(bits)
    template_bits = [int(c) for c in template]
    # NIST usa bloques de longitud 1032 y cuenta solapada
    # Versión simplificada: contamos coincidencias solapadas en toda la secuencia
    count = 0
    for i in range(n - m + 1):
        if bits[i:i+m] == template_bits:
            count += 1
    # Valor esperado para secuencia aleatoria: (n - m + 1) / 2^m
    mu = (n - m + 1) / (2**m)
    var = (n - m + 1) * (1/(2**m) - (2*m - 1)/(2**(2*m)))
    if var <= 0:
        return 0.5
    z = (count - mu) / math.sqrt(var)
    p = erfc(abs(z) / math.sqrt(2))
    return p

# 9. Maurer's Universal Test
def maurer_universal_test(bits, L=8):
    n = len(bits)
    Q = 10 * 2**L
    K = n // L - Q
    if K <= 0:
        return None
    # tabla de últimas posiciones
    table = {}
    # Fase de inicialización
    for i in range(Q):
        block = bits[i*L:(i+1)*L]
        val = 0
        for b in block:
            val = (val << 1) | b
        table[val] = i + 1   # posición 1-indexada
    # Fase de prueba
    sum_log = 0.0
    for j in range(Q, Q+K):
        block = bits[j*L:(j+1)*L]
        val = 0
        for b in block:
            val = (val << 1) | b
        if val in table:
            dist = j + 1 - table[val]
            sum_log += math.log2(dist)
        else:
            # Primera aparición: se usa la distancia máxima
            sum_log += math.log2(j + 1)
        table[val] = j + 1
    f_u = sum_log / K
    # Valores esperados para L=8 (NIST)
    mu = 7.1836656
    sigma = 0.852549
    p = erfc(abs(f_u - mu) / (sigma * math.sqrt(2)))
    return p

# 10. Linear Complexity Test (Berlekamp-Massey)
def berlekamp_massey(s):
    """Implementación BM sobre GF(2)"""
    n = len(s)
    C = [1]
    B = [1]
    L = 0
    m = 1
    b = 1
    for i in range(n):
        # calcular discrepancia d
        d = s[i]
        for j in range(1, L+1):
            d ^= C[j] * s[i-j]
        if d == 0:
            m += 1
        else:
            T = C[:]
            # Asegurar longitud
            while len(C) < len(B) + m:
                C.append(0)
            for j in range(len(B)):
                C[j+m] ^= B[j]
            if 2*L <= i:
                L = i + 1 - L
                B = T
                b = d
                m = 1
            else:
                m += 1
    return L

def linear_complexity_test(bits, M=500):
    n = len(bits)
    N = n // M
    if N == 0:
        return None
    comps = []
    for i in range(N):
        block = bits[i*M:(i+1)*M]
        comps.append(berlekamp_massey(block))
    # Media teórica para secuencia aleatoria: M/2
    # NIST usa un test chi-cuadrado con 6 grados de libertad
    # Versión simplificada:
    mean_exp = M/2.0
    chi_sq = 0.0
    for c in comps:
        chi_sq += (c - mean_exp)**2 / mean_exp
    p = stats.chi2.sf(chi_sq, N)
    return p

# 11. Serial Test
def serial_test(bits, m=16):
    n = len(bits)
    if n < m:
        return None
    padded = bits + bits[:m-1]
    def count_patterns(length):
        patterns = {}
        for i in range(n):
            pat = 0
            for j in range(length):
                pat = (pat << 1) | padded[i+j]
            patterns[pat] = patterns.get(pat, 0) + 1
        return patterns
    count_m = count_patterns(m)
    count_m1 = count_patterns(m-1)
    psi_m = sum(cnt*(cnt-1) for cnt in count_m.values()) / (n*(n-1)) * (2**m)
    psi_m1 = sum(cnt*(cnt-1) for cnt in count_m1.values()) / (n*(n-1)) * (2**(m-1))
    delta = psi_m - psi_m1
    p = stats.chi2.sf(delta, 2**(m-1))
    return p

# 12. Approximate Entropy Test
def approximate_entropy_test(bits, m=10):
    n = len(bits)
    if n < m:
        return None
    padded = bits + bits[:m]
    def entropy(m_val):
        counts = {}
        for i in range(n):
            pat = 0
            for j in range(m_val):
                pat = (pat << 1) | padded[i+j]
            counts[pat] = counts.get(pat, 0) + 1
        sum_ = 0.0
        for cnt in counts.values():
            p = cnt / n
            sum_ += p * math.log(p)
        return -sum_
    phi_m = entropy(m)
    phi_m1 = entropy(m+1)
    apen = phi_m - phi_m1
    chi_sq = 2 * n * (math.log(2) - apen)
    p = stats.chi2.sf(chi_sq, 2**m)
    return p

# 13. Cumulative Sums Test (forward)
def cumulative_sums_test(bits):
    n = len(bits)
    S = 0
    maxS = 0
    for b in bits:
        S += 1 if b == 1 else -1
        if abs(S) > maxS:
            maxS = abs(S)
    z = maxS
    # Calcular p-value usando aproximación normal
    # Fórmula simplificada usada por NIST
    sum_ = 0.0
    for k in range(-z+1, z):
        term1 = stats.norm.cdf((4*k+1)*z / math.sqrt(n))
        term2 = stats.norm.cdf((4*k-1)*z / math.sqrt(n))
        sum_ += term1 - term2
    p = 1 - sum_
    return p

# 14. Random Excursions Test (simplificada)
def random_excursions_test(bits):
    # Construir caminata aleatoria
    walk = [0]
    for b in bits:
        walk.append(walk[-1] + (1 if b==1 else -1))
    # Encontrar ciclos entre ceros
    cycles = []
    start = 0
    for i in range(1, len(walk)):
        if walk[i] == 0:
            if i > start:
                cycles.append(walk[start:i+1])
            start = i
    if not cycles:
        return 0.5
    # Estados a analizar: -5..-1, 1..5
    states = list(range(-5,0)) + list(range(1,6))
    chi_sq = 0.0
    for state in states:
        observed = sum(cycle.count(state) for cycle in cycles)
        expected = len(cycles) * 0.5  # aproximado
        if expected > 0:
            chi_sq += (observed - expected)**2 / expected
    p = stats.chi2.sf(chi_sq, len(states)-1)
    return p

# 15. Random Excursions Variant Test (simplificada)
def random_excursions_variant_test(bits):
    walk = [0]
    for b in bits:
        walk.append(walk[-1] + (1 if b==1 else -1))
    # Estados -9..-1, 1..9
    states = list(range(-9,0)) + list(range(1,10))
    chi_sq = 0.0
    for state in states:
        observed = walk.count(state)
        expected = len(walk) * 0.5  # aproximado
        if expected > 0:
            chi_sq += (observed - expected)**2 / expected
    p = stats.chi2.sf(chi_sq, len(states)-1)
    return p

# ------------------------------------------------------------
# Función principal: ejecuta todas las pruebas
def run_all_tests(bits):
    results = {}
    results['Monobit'] = monobit_test(bits)
    results['Block Frequency'] = block_frequency_test(bits, M=128)
    results['Runs'] = runs_test(bits)
    results['Longest Run of Ones'] = longest_run_ones_test(bits, M=10000)
    results['Binary Matrix Rank'] = binary_matrix_rank_test(bits, 32, 32)
    results['DFT'] = dft_test(bits)
    results['Non-overlapping Template'] = non_overlapping_template_test(bits)
    results['Overlapping Template'] = overlapping_template_test(bits)
    results['Maurer Universal'] = maurer_universal_test(bits, L=8)
    results['Linear Complexity'] = linear_complexity_test(bits, M=500)
    results['Serial'] = serial_test(bits, m=16)
    results['Approximate Entropy'] = approximate_entropy_test(bits, m=10)
    results['Cumulative Sums'] = cumulative_sums_test(bits)
    results['Random Excursions'] = random_excursions_test(bits)
    results['Random Excursions Variant'] = random_excursions_variant_test(bits)
    return results

# ------------------------------------------------------------
# Generar secuencia de prueba
if __name__ == "__main__":
    import random
    random.seed(500)
    n_bits = 1000000
    bits = [random.getrandbits(1) for _ in range(n_bits)]

    p_vals = run_all_tests(bits)
    alpha = 0.01
    print("NIST Statistical Test Suite Results (α=0.01)")
    print("Sequence length:", n_bits)
    print("-" * 55)
    for name, p in p_vals.items():
        if p is None:
            print(f"{name:30} : SKIPPED")
        else:
            result = "PASS" if p >= alpha else "FAIL"
            print(f"{name:30} : p = {p:.6f}  -> {result}")

NIST Statistical Test Suite Results (α=0.01)
Sequence length: 1000000
-------------------------------------------------------
Monobit                        : p = 0.020888  -> PASS
Block Frequency                : p = 0.272778  -> PASS
Runs                           : p = 0.501159  -> PASS
Longest Run of Ones            : p = 0.122557  -> PASS
Binary Matrix Rank             : p = 0.408469  -> PASS
DFT                            : p = 0.927617  -> PASS
Non-overlapping Template       : p = 0.660578  -> PASS
Overlapping Template           : p = 0.332518  -> PASS
Maurer Universal               : p = 0.995714  -> PASS
Linear Complexity              : p = 1.000000  -> PASS
Serial                         : p = 1.000000  -> PASS
Approximate Entropy            : p = 0.000000  -> FAIL
Cumulative Sums                : p = 0.016666  -> PASS
Random Excursions              : p = 0.000000  -> FAIL
Random Excursions Variant      : p = 0.000000  -> FAIL


In [9]:
!pip install nistrng

In [10]:
import random
import numpy as np
from nistrng import *

def main():
    # Generar secuencia de bits (1,000,000 bits)
    bit_length = 100_000
    print(f"Generating {bit_length} random bits...")
    bits = [random.getrandbits(1) for _ in range(bit_length)]
    # Usar tipo con signo (int8) para evitar desbordamiento en DFT
    binary_sequence = np.array(bits, dtype=np.int8)

    # Verificar pruebas aplicables
    eligible_battery = check_eligibility_all_battery(binary_sequence, SP800_22R1A_BATTERY)

    print("\n[!] Starting NIST tests...")
    print(f"Eligible tests: {len(eligible_battery)}")
    for name in eligible_battery.keys():
        print(f"  - {name}")

    # Ejecutar todas las pruebas
    results = run_all_battery(binary_sequence, eligible_battery, False)

    # Mostrar resultados
    passed = 0
    print("\n" + "="*60)
    print("NIST Statistical Test Results")
    print("="*60)
    for result, elapsed in results:
        status = "[PASS]" if result.passed else "[FAIL]"
        print(f"{status} {result.name:35} : p = {result.score:.6f}  (time: {elapsed:.2f} ms)")
        if result.passed:
            passed += 1
    print("="*60)
    print(f"Final: {passed}/{len(results)} tests passed")
    if passed == len(results):
        print("Python random PASSED all NIST tests.")
    else:
        print("Python random FAILED some tests.")

if __name__ == "__main__":
    main()

Generating 100000 random bits...

[!] Starting NIST tests...
Eligible tests: 12
  - monobit
  - frequency_within_block
  - runs
  - longest_run_ones_in_a_block
  - binary_matrix_rank
  - dft
  - non_overlapping_template_matching
  - serial
  - approximate_entropy
  - cumulative sums
  - random_excursion
  - random_excursion_variant

NIST Statistical Test Results
[PASS] Monobit                             : p = 0.342782  (time: 0.00 ms)
[PASS] Frequency Within Block              : p = 0.843265  (time: 0.00 ms)
[PASS] Runs                                : p = 0.240583  (time: 24.00 ms)
[PASS] Longest Run Ones In A Block         : p = 0.738145  (time: 1.00 ms)
[PASS] Binary Matrix Rank                  : p = 0.914976  (time: 198.00 ms)
[FAIL] Discrete Fourier Transform          : p = 0.000000  (time: 5.00 ms)
[FAIL] Non Overlapping Template Matching   : p = 0.000000  (time: 489.00 ms)
[FAIL] Serial                              : p = 0.000000  (time: 9705.00 ms)
[FAIL] Approximate Entropy 

In [11]:
import numpy as np
import secrets
from nistrng import *

def generar_bits_seguros(bit_length):

    n_bytes = (bit_length + 7) // 8
    bytes_aleatorios = secrets.token_bytes(n_bytes)
    array_bytes = np.frombuffer(bytes_aleatorios, dtype=np.uint8)
    bits = np.unpackbits(array_bytes)
    return bits[:bit_length]

def main():
    bit_length = 100_000
    print(f"Generating {bit_length} random bits with secrets...")

    bits = generar_bits_seguros(bit_length)

    binary_sequence = np.array(bits, dtype=np.int8)

    eligible_battery = check_eligibility_all_battery(binary_sequence, SP800_22R1A_BATTERY)

    print("\n[!] Starting NIST tests...")
    print(f"Eligible tests: {len(eligible_battery)}")
    for name in eligible_battery.keys():
        print(f"  - {name}")

    results = run_all_battery(binary_sequence, eligible_battery, False)

    passed = 0
    print("\n" + "="*60)
    print("NIST Statistical Test Results")
    print("="*60)
    for result, elapsed in results:
        status = "[PASS]" if result.passed else "[FAIL]"
        print(f"{status} {result.name:35} : p = {result.score:.6f}  (time: {elapsed:.2f} ms)")
        if result.passed:
            passed += 1
    print("="*60)
    print(f"Final: {passed}/{len(results)} tests passed")
    if passed == len(results):
        print("Python secrets PASSED all NIST tests.")
    else:
        print("Python secrets FAILED some tests.")

if __name__ == "__main__":
    main()

Generating 100000 random bits with secrets...

[!] Starting NIST tests...
Eligible tests: 12
  - monobit
  - frequency_within_block
  - runs
  - longest_run_ones_in_a_block
  - binary_matrix_rank
  - dft
  - non_overlapping_template_matching
  - serial
  - approximate_entropy
  - cumulative sums
  - random_excursion
  - random_excursion_variant

NIST Statistical Test Results
[PASS] Monobit                             : p = 0.257594  (time: 0.00 ms)
[PASS] Frequency Within Block              : p = 0.220421  (time: 0.00 ms)
[PASS] Runs                                : p = 0.223078  (time: 26.00 ms)
[PASS] Longest Run Ones In A Block         : p = 0.502533  (time: 1.00 ms)
[PASS] Binary Matrix Rank                  : p = 0.531660  (time: 214.00 ms)
[FAIL] Discrete Fourier Transform          : p = 0.000000  (time: 5.00 ms)
[PASS] Non Overlapping Template Matching   : p = 0.288339  (time: 614.00 ms)
[FAIL] Serial                              : p = 0.000000  (time: 4268.00 ms)
[FAIL] Approxi